# 07c · Weather: ERA5 pressure levels from ARCO-ERA5

Wind, temperature, relative humidity and height at **925, 850, 700, 600 and 500 hPa** (about 750, 1500, 3000, 4200 and 5600 m) for every site, read from **ARCO-ERA5** on Google Cloud Storage: the same columns as `era5_levels` (`u_850`, `t_700`, `rh_850`, `z_500`, ...), with no CDS queue.

ARCO stores the 37 pressure levels of an hour together, so every read brings about 150 MB for the whole world. To keep it to hours instead of days it reads **every 3rd UTC hour** (`EVERY`) of each site's competition days and the day before, 5 to 21 h local solar time; the air aloft changes slowly and the hours between can be interpolated. Each month is saved to `database/weather/arco/levels/` as soon as it is read, so **if Colab disconnects, Run All again** and it carries on. With this, notebook 07 only needs to ask the CDS for ERA5-Land.

## 1. Settings

In [ ]:
DRIVE_FOLDER_ID = "1_JO0XMMQ4NKVfpeA8uKQRkZZPNC4sqZ7"   # project folder in Google Drive (the id at the end of its link)
THREADS = 12         # fields read at a time (each holds all 37 levels, about 150 MB)
EVERY = 3            # read every 3rd UTC hour; the air aloft changes slowly
SOLAR = "5-21"       # local solar hours kept for each site
REPO_URL = "https://github.com/RobertojonesFAI/Pre-task-analyzer.git"
REPO = "/content/Pre-task-analyzer"

## 2. Mount Drive, get the code

In [ ]:
import pandas as pd
import os, shutil, subprocess
from google.colab import auth, drive
from googleapiclient.discovery import build

auth.authenticate_user()
if not os.path.isdir("/content/drive/MyDrive"):
    drive.mount("/content/drive")
service = build("drive", "v3")
FIELDS = "id, name, parents, driveId, ownedByMe"


def path_up(file_id, root):
    """Folder names from a file up to the top of its drive, joined under root."""
    names, current = [], file_id
    while True:
        meta = service.files().get(fileId=current, fields=FIELDS, supportsAllDrives=True).execute()
        if not meta.get("parents"):
            return os.path.join(root, *reversed(names))
        names.append(meta["name"])
        current = meta["parents"][0]


def drive_path(folder_id):
    """A writable local path for a Drive folder, from its id."""
    meta = service.files().get(fileId=folder_id, fields=FIELDS, supportsAllDrives=True).execute()
    if meta.get("driveId"):  # in a shared drive
        drive_name = service.drives().get(driveId=meta["driveId"]).execute()["name"]
        return path_up(folder_id, f"/content/drive/Shareddrives/{drive_name}")
    if meta.get("ownedByMe"):  # in my own Drive
        return path_up(folder_id, "/content/drive/MyDrive")
    # Shared with me: write through a shortcut to it in my Drive
    page = None
    while True:
        res = service.files().list(
            q="mimeType='application/vnd.google-apps.shortcut' and trashed=false",
            fields="nextPageToken, files(id, shortcutDetails)", pageToken=page).execute()
        for f in res.get("files", []):
            if f.get("shortcutDetails", {}).get("targetId") == folder_id:
                return path_up(f["id"], "/content/drive/MyDrive")
        page = res.get("nextPageToken")
        if not page:
            raise SystemExit(f"'{meta['name']}' is shared with you. Add a shortcut to it in your Drive "
                             "(right click -> Organize -> Add shortcut), then rerun this cell.")


ROOT = drive_path(DRIVE_FOLDER_ID)
DB = os.path.join(ROOT, "database")
assert os.path.isfile(os.path.join(DB, "site_chips.parquet")), "Run notebook 06 first: database/site_chips.parquet is missing"
subprocess.run(["pip", "install", "-q", "gcsfs", "zarr", "netCDF4"], check=True)
print("Database in", DB)

if os.path.exists(REPO):
    subprocess.run(["git", "-C", REPO, "pull", "-q"], check=True)
else:
    subprocess.run(["git", "clone", "-q", REPO_URL, REPO], check=True)


def run(*cmd):
    """Run a script and show its output; stopping the cell also stops the script (otherwise it would
    keep running in the background)."""
    proc = subprocess.Popen(["python3", "-u", *cmd], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    try:
        for line in proc.stdout:
            print(line, end="")
        code = proc.wait()
    except KeyboardInterrupt:
        proc.terminate()
        proc.wait()
        print("Stopped.")
        raise  # also stops Run All
    if code != 0:  # stop Run All here instead of building tables from an unfinished step
        raise RuntimeError(f"{os.path.basename(cmd[0])} stopped with an error (exit code {code}); see the lines above")
    return code

## 3. What there is to read

How many hours and global fields, and whether some hours are newer than ARCO (those come from the CDS). Nothing is read yet.

In [ ]:
run(f"{REPO}/scripts/era5_arco.py", DB, "--product", "levels", "--threads", str(THREADS), "--every", str(EVERY), "--solar", SOLAR, "--plan-only")

## 4. Read

One line per month, with its speed and the time left. Stopping this cell also stops the reading; months already saved are kept.

In [ ]:
run(f"{REPO}/scripts/era5_arco.py", DB, "--product", "levels", "--threads", str(THREADS), "--every", str(EVERY), "--solar", SOLAR)

## 5. Build the table

Merges the months into `weather/era5_levels.parquet` with any files already downloaded from the CDS.

In [ ]:
run(f"{REPO}/scripts/era5.py", DB, "--convert-only", "--tables", "levels")

## 6. A look at the table

In [ ]:
w = pd.read_parquet(os.path.join(DB, "weather", "era5_levels.parquet"))
print(f"{len(w):,} rows, {w.chip_id.nunique():,} chips, {w.time.dt.normalize().nunique():,} days")
print(w.drop(columns=["chip_id", "time"]).describe().T[["mean", "min", "50%", "max"]].round(1))